# AwareLiquid · Phase 5 Backbone Training (Kaggle P100)

Trains MT-LNN residual adapters + LoRA on **TinyLlama-1.1B-Chat-v1.0**, frozen base, on WikiText-2.

**Settings** target a single Kaggle P100 (16GB):
- steps: 1000
- seq_len: 1024
- batch: 1, grad_accum: 8 (effective batch 8)
- adapter every 4th decoder layer
- LoRA on q/k/v/o projections

**Before running**
1. Notebook → Settings → Accelerator → **GPU P100**
2. Notebook → Settings → Internet → **On** (needed for HF + dataset download)
3. Expected runtime: ~3-4 h (well under the 12 h session limit)

Outputs land in `/kaggle/working/` and are downloadable from the right sidebar after the run.

## 1 · Environment sanity check

In [ ]:
import torch, platform
print('python', platform.python_version())
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('mem_gb', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
assert torch.cuda.is_available(), 'GPU is required — enable P100 in notebook settings'

## 2 · Clone the repo into /kaggle/working/

Replace `REPO_URL` with the actual git URL (public HTTPS works; for private repos, use a deploy token URL such as `https://<TOKEN>@github.com/everest-an/O1.git`).

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/everest-an/O1.git'  # ← edit if private
REPO_DIR = '/kaggle/working/O1'
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
subprocess.check_call(['git', 'log', '-1', '--oneline'])

## 3 · Install Python dependencies

In [ ]:
!pip install -q -r requirements.txt accelerate safetensors peft datasets

## 4 · Adapter smoke test

Fast — catches import / shape regressions before the long run.

In [ ]:
!python -m pytest tests/test_llama_adapter.py -q

## 5 · Train MT adapter + LoRA on frozen TinyLlama-1.1B

`scripts/cloud_llama_mt_experiment.sh` does train → PPL ablation → needle ablation. Env vars override defaults to fit P100.

In [ ]:
%env MODEL=TinyLlama/TinyLlama-1.1B-Chat-v1.0
%env SEQ_LEN=1024
%env BATCH=1
%env GRAD_ACCUM=8
%env STEPS=1000
%env MT_EVERY=4
%env NEEDLE_CONTEXTS=1024 2048 4096
%env NEEDLE_SAMPLES=5
%env OUT_DIR=/kaggle/working/checkpoints/llama_mt_adapter
%env RESULT_DIR=/kaggle/working/benchmarks/kaggle_run
!bash scripts/cloud_llama_mt_experiment.sh

## 6 · Package outputs for download

Bundles the adapter checkpoint and benchmark JSONs into a single archive under `/kaggle/working/`, which appears in the notebook's Output panel.

In [ ]:
import shutil, glob
from pathlib import Path
out_root = Path('/kaggle/working/awareliquid_phase5_artifacts')
out_root.mkdir(parents=True, exist_ok=True)
for ckpt in sorted(glob.glob('/kaggle/working/checkpoints/llama_mt_adapter/*.pt'))[-2:]:
    shutil.copy(ckpt, out_root / Path(ckpt).name)
for j in glob.glob('/kaggle/working/benchmarks/kaggle_run/*.json'):
    shutil.copy(j, out_root / Path(j).name)
for j in glob.glob('/kaggle/working/benchmarks/kaggle_run/*.log'):
    shutil.copy(j, out_root / Path(j).name)
archive = shutil.make_archive('/kaggle/working/awareliquid_phase5', 'zip', out_root)
print('archive:', archive)
print('size MB:', round(Path(archive).stat().st_size / 1024**2, 1))

## 7 · Quick eyeball of results

In [ ]:
import json, pathlib
for name in ('ppl_ablation.json', 'needle.json'):
    p = pathlib.Path('/kaggle/working/benchmarks/kaggle_run') / name
    if p.exists():
        print(f'--- {name} ---')
        print(json.dumps(json.loads(p.read_text()), indent=2)[:2000])